# Agent Evaluation

NOTE:: I can not evaluate my agent since with ollama I tried only 2 models and I do not get intermediate_trajectories and this means that the agent does not use the search tool

NOTE: 
either I use manuall agent or I use kestra, or openAI
TODO:  TRY loading another model from ollama and check the results

source: https://www.youtube.com/watch?v=lbEj3Waxs1U&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=45

For RAG, we used the A->Q->A' setup:

A = original answer in the FAQ
Q = generated question from this answer
A' = answer produced by our RAG system

* For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.


* We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

1. Given the ground_truth data set that is generated by RAGBase,
2. for each record,  get the A' (answer from agent that has memory),
3. save the A', the original answer A, the question, and document to question_with_agent_and_human_answers_data.csv
4. create an judge_agent to judge (score, and resoning),
5. use search_text since we find the optimal parameters

# NOTE 1:::: the agent with memory is more expensive that agent without memory
# NOTE 2::::  I will use agent without memory

In [1]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index
import re

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import (load_faq_data,
                    build_index,
                    text_search
)
# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")
# convert the df to dictionary

# note some answers do not have 5 questions
print(f'The number of ground_truth = {len(ground_truth)}')

# create search index and fit with document (note when I fit, I do not need to use the optimal parameters, only for search I need to use the optimal parameters)
search_index = build_index(documents_llm)

# # Bind your fitted index to the search function using a lambda
# bound_text_search = lambda query: text_search(query=query, search_index=search_index)

The number of documents = 113

The number of ground_truth = 510

In [2]:
rec = ground_truth[0]
rec

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'document': '489dd1c9d9'}

# Create a lookup table:

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
original_document = doc_idx[rec['document']]
original_document

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

# Agent with a Memory

In [5]:
from smolagents import CodeAgent, LiteLLMModel, Tool
from langchain_core.messages import HumanMessage, AIMessage


class FAQSearchTool(Tool):
    name = "text_search_tuned"
    description = (
        "Searches the LLM Zoomcamp FAQ database using optimized keyword weights. "
        "Use this for any course registration, schedules, or prerequisite questions."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The simple search keywords or queries to pass into the FAQ database."
        }
    }

    output_type = "string"

    def __init__(self, search_index, **kwargs):
        super().__init__(**kwargs)
        self.search_index = search_index

    def forward(self, query: str) -> str:

        boost_dict = {
            "question": 3.0,
            "section": 0.5,
            "answer": 10.0,
        }

        filter_dict = {
            "course": "llm-zoomcamp"
        }
        search_results = self.search_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
        lines = []

        for doc in search_results:
            lines.append(f"Section: {doc['section']}")
            lines.append(f"Q: {doc['question']}")
            lines.append(f"A: {doc['answer']}\n")
        return "\n".join(lines).strip() if lines else "No matching FAQ entries found."


class FAQAgent:
    def __init__(self, search_index):
        # Memory
        self.chat_history = []
    
        # Model
        self.model = LiteLLMModel(
            model_id="ollama_chat/qwen2.5:3b-instruct",
            api_base="http://127.0.0.1:11434"
        )
        # Tool
        self.faq_tool = FAQSearchTool(
            search_index=search_index
        )
        # Agent
        self.agent = CodeAgent(
            model=self.model,
            tools=[self.faq_tool],
            max_steps=5,
            verbosity_level=0
        )

    def get_history(self):

        if not self.chat_history:
            return "No previous conversation."

        history = []
        for message in self.chat_history:
            if isinstance(message, HumanMessage):
                history.append(
                    f"User: {message.content}"
                )
            elif isinstance(message, AIMessage):
                history.append(
                    f"Assistant: {message.content}"
                )
        return "\n".join(history)
    
    def clear_memory(self):
        """I want to clear memory after each query to reduce the cost"""
        self.chat_history = []

    def update_memory(self, question, answer):
        self.chat_history.append(
            HumanMessage(content=question)
        )
        self.chat_history.append(
            AIMessage(content=answer)
        )
        # keep last 10 messages
        self.chat_history = self.chat_history[-10:]

    def run(self, question):
        history = self.get_history()
        task_prompt = f"""
You are an FAQ assistant for LLM Zoomcamp.
Conversation history:
{history}

You have one tool:
text_search_tuned(query)

Follow these steps:
1. Call text_search_tuned with the user's question.
2. Read the returned FAQ information.
3. Answer ONLY using the returned information.
4. Do not copy the search results.
5. Keep the answer concise.

IMPORTANT:
Your final output MUST be inside a code block exactly like this:

<code>
final_answer("your answer here")
</code>

Never write final_answer outside the code block.

User question:
{question}
"""
        result = self.agent.run(
            task_prompt,
            return_full_result=True
        )
        answer = result.output

        # Update memory
        self.update_memory(
            question,
            answer
        )
        return result
    

def get_query_cost_by_agent_with_memeory(agent_result):
    usage_agent_result = {
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
    }
    for step in agent_result.steps:
        if isinstance(step, dict):
            step_usage = step.get("token_usage")
            if step_usage:
                for key in usage_agent_result:
                    usage_agent_result[key] += step_usage.get(key, 0)
    return(usage_agent_result)



def get_tool_call_by_agent_with_memeory(agent_result):
    tool_calls = []

    for step in agent_result.steps:
        if not isinstance(step, dict):
            continue
        code = step.get("code_action")

        if not code:
            continue
        matches = re.findall(
            r'(\w+)\((.*?)\)',
            code,
            re.DOTALL
        )
        for name, args in matches:
            if name != "print":
                tool_calls.append(
                    {
                        "tool": name,
                        "arguments": args
                    }
                )
    return tool_calls



faq_agent_with_memory = FAQAgent(search_index)


question = rec['question']
# I will clear the history to create agent withut memory since I want to reduce the cost
faq_agent_with_memory.clear_memory()
search_index_agent_result = faq_agent_with_memory.run(question)

print("Final answer:")
print(search_index_agent_result.output)


Final answer:

The Zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit 
questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements 
channel on Telegram and Slack before it begins.

In [6]:
usage_agent_result = get_query_cost_by_agent_with_memeory(search_index_agent_result)
usage_agent_result

{'input_tokens': 5463, 'output_tokens': 112, 'total_tokens': 5575}

In [7]:
# get tool_call (agent trajectories)
tool_calls = get_tool_call_by_agent_with_memeory(search_index_agent_result)
tool_calls

[{'tool': 'text_search_tuned',
  'arguments': 'query="Where do I find the Zoom link for Office Hours?"'},
 {'tool': 'final_answer',
  'arguments': '"The Zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live'}]

### For a Document, report its id, question, original and agent answer, cost and tool_calls



In [8]:
def generate_agent_answer(record_from_ground_truth):
    original_document = doc_idx[record_from_ground_truth['document']]

    # I will create  an instance each time since I want agent without a amemory
    faq_agent_with_memory = FAQAgent(search_index)
    faq_agent_with_memory.clear_memory() # to be sure that its memory cleared although I am not si-ure about that 100 percent
    
    question = original_document['question']
    search_index_agent_result = faq_agent_with_memory.run(question)
    answer_agent = search_index_agent_result.output
    tool_calls = get_tool_call_by_agent_with_memeory(search_index_agent_result)
    usage_agent_result = get_query_cost_by_agent_with_memeory(search_index_agent_result)

    record_from_ground_truth_to_report = {
        "question": question,
        "answer_agent": answer_agent,
        "answer_orig": original_document["answer"],
        "tool_calls": tool_calls,
        "cost": usage_agent_result,
        "document": original_document['id'],
    }
    return record_from_ground_truth_to_report

generate_agent_answer(ground_truth[0])

{'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer_agent': 'The zoom link for Office Hours or live/workshop sessions is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins, and also available on the DataTalksClub YouTube Channel.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very acti

In [10]:
len(ground_truth)

510

# Run in Parallel

In [11]:
# note:  I will generate only the first 5, since when I run for all the gorund truth, I get an error
from evaluation_utils import  run_in_parallel
agent_answers= run_in_parallel(
    func=generate_agent_answer,
    items=ground_truth[0:5],
)

Processing: 100%|██████████| 5/5 [00:11<00:00,  2.29s/it]


In [12]:
df_agent = pd.DataFrame(agent_answers)

In [13]:
df_agent.to_csv("./data/agent_answers.csv", index=False)

# Judging answers and trajectories

In [14]:
df_agent = pd.read_csv("./data/agent_answers.csv")
df_agent

,question,answer_agent,answer_orig,tool_calls,cost,document
0,What is the video/zoom link to the stream for ...,The zoom link is only published to instructors...,The zoom link is only published to instructors...,"[{'tool': 'text_search_tuned', 'arguments': 'q...","{'input_tokens': 5529, 'output_tokens': 130, '...",489dd1c9d9
1,What is the video/zoom link to the stream for ...,The zoom link for the 'Office Hours' or live/w...,The zoom link is only published to instructors...,"[{'tool': 'text_search_tuned', 'arguments': 'q...","{'input_tokens': 5381, 'output_tokens': 100, '...",489dd1c9d9
2,What is the video/zoom link to the stream for ...,The zoom link is only published to instructors...,The zoom link is only published to instructors...,"[{'tool': 'text_search_tuned', 'arguments': 'q...","{'input_tokens': 5529, 'output_tokens': 77, 't...",489dd1c9d9
3,What is the video/zoom link to the stream for ...,The zoom link is only published to instructors...,The zoom link is only published to instructors...,"[{'tool': 'text_search_tuned', 'arguments': 'q...","{'input_tokens': 5419, 'output_tokens': 142, '...",489dd1c9d9
4,What is the video/zoom link to the stream for ...,The zoom link is only published to instructors...,The zoom link is only published to instructors...,"[{'tool': 'text_search_tuned', 'arguments': 'q...","{'input_tokens': 5458, 'output_tokens': 121, '...",489dd1c9d9


In [15]:
agent_answers = df_agent.to_dict(orient = 'records')
agent_answers

[{'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer_agent': 'The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.',
  'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
  'tool_calls': '[{\'tool\': 

In [16]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

# Define the judge function:

In [17]:
rec_agent_answer = agent_answers[0]
rec_agent_answer

{'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer_agent': 'The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'tool_calls': '[{\'tool\': \'te

In [ ]:
prompt = agent_judge_prompt.format(
    question=rec_agent_answer["question"],
    answer_orig=rec_agent_answer["answer_orig"],
    answer_agent=rec_agent_answer["answer_agent"],
    tool_calls=json.dumps(tool_calls, indent=2),
)

from evaluation_utils import llm_structured_retry
result, usage = llm_structured_retry(
    instructions=agent_judge_instructions,
    user_prompt=prompt,
    output_type=AgentEvaluation,
)

print(f'result={result}')
print(f'usage={usage}')

result=answer_reasoning='The agent provided the correct answer, which stated that the zoom link is only available 
for instructors/presenters/TAs. It also included other relevant information about how students can participate and 
watch the sessions via YouTube Live and Slido.' answer_score='good' trajectory_reasoning='The agent started with a 
text search to find the Zoom link, which was the core of the question. Then it provided the final answer directly. 
The number of tool calls was reasonable (1). The query was relevant and included important keywords from the 
question.' trajectory_score='good'

usage={'input_tokens': 576, 'output_tokens': 128, 'total_tokens': 704}

In [ ]:
MODEL="gemma3:4b"
MAX_RETRIES=3

def evaluate_agent_answer(rec, model=MODEL, max_retries=MAX_RETRIES):
    tool_calls = rec["tool_calls"]

    # if isinstance(tool_calls, str):
    #     tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
    instructions=agent_judge_instructions,
    user_prompt=prompt,
    output_type=AgentEvaluation,
    model=model,
    max_retries=max_retries
)

    return result, usage

# test the function
agent_eval, usage = evaluate_agent_answer(agent_answers[0])
print(agent_eval)
print(usage)

AgentEvaluation(
    answer_reasoning='The agent provided the same answer as the ground truth, which is that the zoom link is only 
published to instructors/presenters/TAs. The agent also included additional information about how students can 
participate via YouTube Live and submit questions to Slido.',
    answer_score='good',
    trajectory_reasoning='The agent started with a direct answer based on the provided information, which is good. 
It then expanded upon this by providing additional details about alternative participation methods (YouTube Live, 
Slido). The tool calls were relevant and appropriately used to retrieve the core information. The number of tool 
calls was reasonable for this question.',
    trajectory_score='good'
)

{'input_tokens': 570, 'output_tokens': 142, 'total_tokens': 712}

# Running the agent judge

In [ ]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

# tes the fuction
result, usage = judge_agent_record(agent_answers[0])

print(result)
print(usage)

{
    'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
    'document': '489dd1c9d9',
    'answer_score': 'good',
    'answer_reasoning': 'The agent provided the same answer as the ground truth, which is that the zoom link is 
only published to instructors/presenters/TAs. The agent also included additional information about how students can
participate via YouTube Live and submit questions to Slido.',
    'trajectory_score': 'good',
    'trajectory_reasoning': 'The agent started with a direct answer based on the provided information, which is 
good. It then expanded upon this by providing additional details about alternative participation methods (YouTube 
Live, Slido). The tool calls were relevant and appropriately used to retrieve the core information. The number of 
tool calls was reasonable for this question.'
}

{'input_tokens': 570, 'output_tokens': 142, 'total_tokens': 712}

# Run in Parallel

In [28]:
results_agent_evaluation = run_in_parallel(
    func=judge_agent_record,
    items=agent_answers,
)

Processing: 100%|██████████| 5/5 [00:18<00:00,  3.70s/it]


In [29]:
agent_evaluations = []
usages = []

for evaluation, usage in results_agent_evaluation:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [33]:
df_agent_eval = pd.DataFrame(agent_evaluations)

# save the df
df_agent_eval.to_csv("./data/agent-evaluations.csv", index=False)

# Check the answer scores:

In [32]:
df_agent_eval["answer_score"].value_counts(normalize=True)

answer_score
good    0.8
bad     0.2
Name: proportion, dtype: float64

# Check the trajectory scores:

In [34]:
df_agent_eval["trajectory_score"].value_counts(normalize=True)

trajectory_score
good    1.0
Name: proportion, dtype: float64